# 🧪 LocateAnything-3B · Test đếm SẢN PHẨM (nhanh, vài query KHÓ)

Test **open-vocab** của LocateAnything-3B trên **3 video thật**:

| Video | Bài toán | Vạch |
|---|---|---|
| `packages_rollers.mp4` | đếm **package** (băng chuyền con lăn) | NGANG y≈0.65 |
| `packages_belt.mp4` | đếm **package** (băng chuyền có nhãn) | NGANG y≈0.60 |
| `tomatoes_sorting.mp4` | đếm **cà chua** (dây chuyền phân loại) | NGANG y≈0.72 |

Vật đều đi **xuống/về phía camera** → dùng **vạch NGANG**. Mỗi video chạy vài
**query KHÓ** (mô tả bằng lời + tiếng Việt) để xem sức hiểu ngôn ngữ của model.

Tối ưu tốc độ: ảnh nhỏ (640) · ít token (256) · lấy mẫu thưa · **nạp model 1 LẦN**
rồi chạy hết. Video đã nằm sẵn trong repo → không cần upload.

In [ ]:
# ⚙️ Cài đặt: clone repo (kèm video) + phụ thuộc + kiểm tra GPU
import os, sys, subprocess

def sh(*a):
    print("$", " ".join(a)); subprocess.run(list(a), check=True)

if os.path.isdir("/kaggle/working"): WORK = "/kaggle/working"
elif os.path.isdir("/content"):      WORK = "/content"
else:                                 WORK = os.getcwd()
os.chdir(WORK); print("WORK =", WORK)

BRANCH = "claude/locate-anything-test-suite-xwju2f"
URL    = "https://github.com/nguyendinhhuyht20032004-ai/VisionOS.git"
REPO   = os.path.join(WORK, "VisionOS")
if not os.path.isdir(os.path.join(REPO, ".git")):
    sh("git", "clone", "--depth", "1", "--branch", BRANCH, URL, REPO)
else:
    sh("git", "-C", REPO, "fetch", "--depth", "1", "origin", BRANCH)
    sh("git", "-C", REPO, "reset", "--hard", "origin/" + BRANCH)

CODE = os.path.join(REPO, "VisionOS")          # code + sample_videos/ ở đây
os.chdir(CODE); sys.path.insert(0, CODE)
print("CODE =", CODE)

# LocateAnything-3B CẦN transformers==4.57.1
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "transformers==4.57.1", "accelerate", "supervision",
                "eva-decord", "lmdb"], check=False)

# xoá cache module (để chạy lại lấy code mới nhất)
for m in [m for m in list(sys.modules)
          if m.split(".")[0] in ("la_counting", "recognition", "run_la_conveyor")]:
    del sys.modules[m]

try:
    import torch
    print("CUDA:", torch.cuda.is_available(),
          torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
except Exception as e:
    print("torch:", e)

In [ ]:
# 🎯 Cấu hình 3 video + query KHÓ + xem trước VẠCH (không cần GPU)
import cv2, numpy as np, matplotlib.pyplot as plt

VID = os.path.join(CODE, "sample_videos")
VIDEOS = [
  {"name": "Package · con lăn", "task": "package",
   "path": os.path.join(VID, "packages_rollers.mp4"),
   "orient": "horizontal", "line_pos": 0.65,
   "queries": ["a cardboard box on the roller conveyor",
               "a sealed shipping package",
               "kiện hàng carton"]},
  {"name": "Package · có nhãn", "task": "package",
   "path": os.path.join(VID, "packages_belt.mp4"),
   "orient": "horizontal", "line_pos": 0.60,
   "queries": ["a cardboard box with a shipping label",
               "a package with a barcode",
               "kiện hàng trên băng chuyền"]},
  {"name": "Cà chua · phân loại", "task": "tomato",
   "path": os.path.join(VID, "tomatoes_sorting.mp4"),
   "orient": "horizontal", "line_pos": 0.72,
   "queries": ["a ripe red tomato",
               "an unripe tomato",
               "cà chua"]},
]

# NÚM tốc độ — chỉnh để nhanh hơn (giảm) hay kỹ hơn (tăng)
PROC_WIDTH     = 640    # bề rộng xử lý (nhỏ = nhanh)
MAX_FRAMES     = 16     # số frame mỗi lượt
STRIDE         = 3      # lấy mỗi N frame
MAX_NEW_TOKENS = 256    # token sinh tối đa (nhỏ = nhanh)

def _frame_with_line(v, frac=0.5):
    cap = cv2.VideoCapture(v["path"]); n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(n * frac)); ok, fr = cap.read(); cap.release()
    if not ok: return None
    h, w = fr.shape[:2]
    if v["orient"] == "horizontal":
        y = int(v["line_pos"] * h); cv2.line(fr, (0, y), (w, y), (0, 0, 255), 3)
    else:
        x = int(v["line_pos"] * w); cv2.line(fr, (x, 0), (x, h), (0, 0, 255), 3)
    return cv2.cvtColor(fr, cv2.COLOR_BGR2RGB)

fig, axes = plt.subplots(1, len(VIDEOS), figsize=(16, 4))
for ax, v in zip(np.atleast_1d(axes), VIDEOS):
    im = _frame_with_line(v)
    if im is not None: ax.imshow(im)
    ax.set_title(v["name"] + "\n(vạch đỏ)"); ax.axis("off")
plt.tight_layout(); plt.show()
print("→ Vạch đỏ nằm đúng dòng chảy vật chưa? Lệch thì sửa line_pos ở trên rồi chạy lại cell này.")

In [ ]:
# 🧠 Nạp LocateAnything-3B MỘT LẦN (lần đầu tải ~6GB, hơi lâu)
from run_la_conveyor import build_fast_detector
detector = build_fast_detector("nvidia/LocateAnything-3B",
                               max_new_tokens=MAX_NEW_TOKENS, iou=0.5, max_boxes=60)
print("✅ Model sẵn sàng — chạy cell tiếp theo để đếm.")

In [ ]:
# ▶️ Chạy đếm: mỗi (video × query) một lượt → scorecard + frame annotate
import time, math
from run_la_conveyor import run_video

os.makedirs("out_annot", exist_ok=True)
print(f"{'video':22}{'query':42}{'qua vạch':>9}{'det/fr':>8}{'giây':>7}")
print("-" * 88)
shots = []
t_all = time.time()
for v in VIDEOS:
    for q in v["queries"]:
        stem  = os.path.splitext(os.path.basename(v["path"]))[0]
        qsafe = "".join(c if c.isalnum() else "_" for c in q)[:24]
        save  = f"out_annot/{stem}__{qsafe}.jpg"
        t0 = time.time()
        r = run_video(detector, v["path"], q, orient=v["orient"], line_pos=v["line_pos"],
                      proc_width=PROC_WIDTH, max_frames=MAX_FRAMES, stride=STRIDE,
                      save_annotated=save)
        print(f"{v['name'][:21]:22}{q[:41]:42}{r.total_crossings:>9}"
              f"{r.avg_detections:>8.1f}{time.time()-t0:>7.1f}")
        shots.append((f"{v['task']} · {q}", save))
print("-" * 88)
print(f"Xong {len(shots)} lượt trong {time.time()-t_all:.0f}s. "
      f"'qua vạch' = số vật cắt vạch · 'det/fr' = box TB/frame (đã NMS).")

# hiển thị frame nhiều box nhất mỗi lượt
cols = 3; nrows = math.ceil(len(shots) / cols)
fig, axes = plt.subplots(nrows, cols, figsize=(16, 4 * nrows))
axf = np.atleast_1d(axes).flat
for ax, (title, path) in zip(axf, shots):
    if os.path.exists(path):
        ax.imshow(cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB))
    ax.set_title(title[:50], fontsize=9); ax.axis("off")
for ax in list(axf)[len(shots):]:
    ax.axis("off")
plt.tight_layout(); plt.show()

### Đọc kết quả
- **`det/fr`** > 0 nghĩa là LocateAnything **hiểu** query và bắt được vật (kể cả mô tả
  khó / tiếng Việt) — đây là điểm mạnh open-vocab.
- **`qua vạch`** = số vật cắt vạch trong đoạn đã chạy (ngắn nên số nhỏ là bình thường).
  Muốn nhiều hơn: tăng `MAX_FRAMES`, giảm `STRIDE` ở cell cấu hình.
- Nếu một query ra **0**: thử diễn đạt cụ thể hơn, hoặc kiểm tra vạch (cell preview).
- LocateAnything **chậm** (~vài giây/frame) → giữ `MAX_FRAMES` nhỏ khi thử query mới.